# Actividad: Patrones de creación — Procesador de informes con Abstract Factory

**Asignatura:** Reutilización de Software (UNIR)
**Estudiante:** Tatiana Betancur Restrepo

Este cuaderno contiene el código Java completo de la actividad y lo compila y ejecuta en Colab. Es autocontenido: no necesita subir archivos ni clonar el repositorio.

| Versión | Ejercicio | Contenido |
|---|---|---|
| `v1-base` | 2 | Categorías ENTRADA y SALIDA; tipos FACTURA y COMPRA |
| `v2-mixta` | 3a | Se añade la categoría MIXTA |
| `v3-pedido` | 3b | Se añade el tipo PEDIDO, solo en ENTRADA |

Las versiones 2 y 3 parten de una copia de la anterior y solo escriben los archivos **nuevos o modificados**, para que se vea el impacto real de cada evolución.

**Uso:** Entorno de ejecución → Ejecutar todas.

## 0. Preparación del entorno

In [ ]:
# Colab trae un JDK instalado; si no lo encuentra, lo instala.
!which javac > /dev/null || (apt-get -qq update && apt-get -qq install -y openjdk-17-jdk-headless > /dev/null)
!javac -version
!rm -rf /content/ProcesadorInformes && mkdir -p /content/ProcesadorInformes/v1-base/src/informes
%cd /content/ProcesadorInformes

## 1. Versión base (ejercicio 2)

Roles del patrón Abstract Factory:

- **Fábrica abstracta:** `FabricaInformes`
- **Fábricas concretas:** `FabricaEntrada`, `FabricaSalida` (una por categoría)
- **Productos abstractos:** `InformeFactura`, `InformeCompra` (extienden `Informe`)
- **Productos concretos:** `FacturaEntrada`, `CompraEntrada`, `FacturaSalida`, `CompraSalida`
- **Cliente:** `ProcesadorInformes`
- **Selección de fábrica y tipo:** enumeraciones `Categoria` y `TipoInforme`

### 1.1 Interfaces y clase base de los productos

In [ ]:
%%writefile v1-base/src/informes/Informe.java
package informes;

/**
 * Interfaz común a todos los informes. El cliente (ProcesadorInformes)
 * solo trabaja con este tipo y nunca con las clases concretas.
 */
public interface Informe {

    String getCategoria();

    String getTipo();

    void procesar(String archivo);
}


In [ ]:
%%writefile v1-base/src/informes/InformeBase.java
package informes;

/**
 * Implementación base compartida por los productos concretos: imprime la
 * cabecera común y delega en cada subclase la acción propia de su
 * combinación categoría + tipo.
 */
public abstract class InformeBase implements Informe {

    private final String categoria;
    private final String tipo;

    protected InformeBase(String categoria, String tipo) {
        this.categoria = categoria;
        this.tipo = tipo;
    }

    @Override
    public String getCategoria() {
        return categoria;
    }

    @Override
    public String getTipo() {
        return tipo;
    }

    @Override
    public final void procesar(String archivo) {
        System.out.println("Procesando informe de " + categoria + " de tipo " + tipo + " [" + archivo + "]");
        System.out.println("    -> " + accion());
    }

    /** Simulación del procesamiento específico del informe. */
    protected abstract String accion();
}


In [ ]:
%%writefile v1-base/src/informes/InformeFactura.java
package informes;

/** Producto abstracto: informe de tipo FACTURA (una variante por categoría). */
public interface InformeFactura extends Informe {
}


In [ ]:
%%writefile v1-base/src/informes/InformeCompra.java
package informes;

/** Producto abstracto: informe de tipo COMPRA (una variante por categoría). */
public interface InformeCompra extends Informe {
}


### 1.2 Productos concretos

In [ ]:
%%writefile v1-base/src/informes/FacturaEntrada.java
package informes;

/** Producto concreto: factura de ENTRADA (recibida de un proveedor). */
public class FacturaEntrada extends InformeBase implements InformeFactura {

    public FacturaEntrada() {
        super("ENTRADA", "FACTURA");
    }

    @Override
    protected String accion() {
        return "Factura recibida de proveedor: se valida el emisor y se registra la cuenta por pagar.";
    }
}


In [ ]:
%%writefile v1-base/src/informes/CompraEntrada.java
package informes;

/** Producto concreto: compra de ENTRADA (mercancía que llega al almacén). */
public class CompraEntrada extends InformeBase implements InformeCompra {

    public CompraEntrada() {
        super("ENTRADA", "COMPRA");
    }

    @Override
    protected String accion() {
        return "Compra recibida: se confirma la recepción de la mercancía y se suma al inventario.";
    }
}


In [ ]:
%%writefile v1-base/src/informes/FacturaSalida.java
package informes;

/** Producto concreto: factura de SALIDA (emitida a un cliente). */
public class FacturaSalida extends InformeBase implements InformeFactura {

    public FacturaSalida() {
        super("SALIDA", "FACTURA");
    }

    @Override
    protected String accion() {
        return "Factura emitida a cliente: se asigna el consecutivo y se registra la cuenta por cobrar.";
    }
}


In [ ]:
%%writefile v1-base/src/informes/CompraSalida.java
package informes;

/** Producto concreto: compra de SALIDA (orden de compra enviada a un proveedor). */
public class CompraSalida extends InformeBase implements InformeCompra {

    public CompraSalida() {
        super("SALIDA", "COMPRA");
    }

    @Override
    protected String accion() {
        return "Orden de compra enviada al proveedor: se compromete el presupuesto y se notifica el pedido.";
    }
}


### 1.3 Fábrica abstracta y fábricas concretas

In [ ]:
%%writefile v1-base/src/informes/FabricaInformes.java
package informes;

/**
 * Fábrica abstracta (Abstract Factory). Declara un método de creación por
 * cada tipo de informe; cada fábrica concreta representa una categoría y
 * devuelve la familia completa de informes de esa categoría.
 */
public interface FabricaInformes {

    InformeFactura crearFactura();

    InformeCompra crearCompra();
}


In [ ]:
%%writefile v1-base/src/informes/FabricaEntrada.java
package informes;

/** Fábrica concreta de la categoría ENTRADA. */
public class FabricaEntrada implements FabricaInformes {

    @Override
    public InformeFactura crearFactura() {
        return new FacturaEntrada();
    }

    @Override
    public InformeCompra crearCompra() {
        return new CompraEntrada();
    }
}


In [ ]:
%%writefile v1-base/src/informes/FabricaSalida.java
package informes;

/** Fábrica concreta de la categoría SALIDA. */
public class FabricaSalida implements FabricaInformes {

    @Override
    public InformeFactura crearFactura() {
        return new FacturaSalida();
    }

    @Override
    public InformeCompra crearCompra() {
        return new CompraSalida();
    }
}


### 1.4 Selección de la fábrica y del método de creación

In [ ]:
%%writefile v1-base/src/informes/Categoria.java
package informes;

/**
 * Asocia el prefijo de categoría del nombre de archivo (CAT_) con su fábrica
 * concreta. Es el único punto del sistema que conoce las fábricas concretas.
 */
public enum Categoria {

    ENT(new FabricaEntrada()),
    SAL(new FabricaSalida());

    private final FabricaInformes fabrica;

    Categoria(FabricaInformes fabrica) {
        this.fabrica = fabrica;
    }

    public FabricaInformes getFabrica() {
        return fabrica;
    }

    public static Categoria desdeCodigo(String codigo) {
        for (Categoria categoria : values()) {
            if (categoria.name().equals(codigo)) {
                return categoria;
            }
        }
        throw new IllegalArgumentException("categoría desconocida '" + codigo + "'");
    }
}


In [ ]:
%%writefile v1-base/src/informes/TipoInforme.java
package informes;

import java.util.function.Function;

/**
 * Asocia el código de tipo del nombre de archivo (_TIPO_) con el método de
 * creación correspondiente de la fábrica abstracta.
 */
public enum TipoInforme {

    FAC(FabricaInformes::crearFactura),
    COM(FabricaInformes::crearCompra);

    private final Function<FabricaInformes, Informe> creador;

    TipoInforme(Function<FabricaInformes, Informe> creador) {
        this.creador = creador;
    }

    public Informe crear(FabricaInformes fabrica) {
        return creador.apply(fabrica);
    }

    public static TipoInforme desdeCodigo(String codigo) {
        for (TipoInforme tipo : values()) {
            if (tipo.name().equals(codigo)) {
                return tipo;
            }
        }
        throw new IllegalArgumentException("tipo de informe desconocido '" + codigo + "'");
    }
}


### 1.5 Cliente y programa principal

In [ ]:
%%writefile v1-base/src/informes/ProcesadorInformes.java
package informes;

/**
 * Cliente del patrón. Interpreta el nombre CAT_TIPO_nombre.txt, obtiene la
 * fábrica de la categoría y le pide el informe del tipo indicado. No conoce
 * ninguna clase concreta, por lo que no cambia cuando el sistema evoluciona.
 */
public class ProcesadorInformes {

    public void procesarInformes(String[] nombres) {
        int procesados = 0;
        int omitidos = 0;
        for (String nombre : nombres) {
            try {
                crearInforme(nombre).procesar(nombre);
                procesados++;
            } catch (IllegalArgumentException | UnsupportedOperationException e) {
                System.out.println("[OMITIDO] " + nombre + ": " + e.getMessage());
                omitidos++;
            }
        }
        System.out.println();
        System.out.println("Resumen: " + procesados + " informe(s) procesado(s), " + omitidos + " omitido(s).");
    }

    Informe crearInforme(String nombre) {
        String[] partes = nombre.split("_", 3);
        if (partes.length < 3 || !partes[2].endsWith(".txt")) {
            throw new IllegalArgumentException("el nombre no sigue el formato CAT_TIPO_nombre.txt");
        }
        FabricaInformes fabrica = Categoria.desdeCodigo(partes[0]).getFabrica();
        return TipoInforme.desdeCodigo(partes[1]).crear(fabrica);
    }
}


In [ ]:
%%writefile v1-base/src/informes/Main.java
package informes;

import java.io.File;
import java.util.Arrays;

/**
 * Prueba del sistema. Sin argumentos usa la lista de ejemplo; con una ruta
 * de carpeta procesa los .txt que contiene; con varios argumentos los toma
 * como nombres de informe.
 */
public class Main {

    public static void main(String[] args) {
        String[] reports = {
            "ENT_FAC_001.txt", "ENT_COM_001.txt", "SAL_FAC_001.txt", "SAL_COM_001.txt",
            "XYZ_FAC_001.txt", "ENT_FAC.txt"
        };
        if (args.length == 1 && new File(args[0]).isDirectory()) {
            reports = new File(args[0]).list((carpeta, nombre) -> nombre.endsWith(".txt"));
            Arrays.sort(reports);
        } else if (args.length > 0) {
            reports = args;
        }
        new ProcesadorInformes().procesarInformes(reports);
    }
}


### 1.6 Compilación y ejecución

Se esperan 4 informes procesados y 2 omitidos (`XYZ_FAC_001.txt`, de categoría desconocida, y `ENT_FAC.txt`, con un formato no válido).

In [ ]:
!cd v1-base && rm -rf out && javac -encoding UTF-8 -d out src/informes/*.java && java -cp out informes.Main

## 2. Evolución 1: categoría MIXTA (ejercicio 3a)

MIXTA es una **familia nueva**. Se crean una fábrica y sus dos productos y se registra el prefijo `MIX` en `Categoria`. La fábrica abstracta, las fábricas existentes y el cliente **no cambian**.

In [ ]:
!rm -rf v2-mixta && cp -r v1-base v2-mixta && rm -rf v2-mixta/out

### 2.1 Clases nuevas

In [ ]:
%%writefile v2-mixta/src/informes/FabricaMixta.java
package informes;

/** Fábrica concreta de la categoría MIXTA (evolución 1). */
public class FabricaMixta implements FabricaInformes {

    @Override
    public InformeFactura crearFactura() {
        return new FacturaMixta();
    }

    @Override
    public InformeCompra crearCompra() {
        return new CompraMixta();
    }
}


In [ ]:
%%writefile v2-mixta/src/informes/FacturaMixta.java
package informes;

/** Producto concreto: factura MIXTA (incluye movimientos de entrada y de salida). */
public class FacturaMixta extends InformeBase implements InformeFactura {

    public FacturaMixta() {
        super("MIXTA", "FACTURA");
    }

    @Override
    protected String accion() {
        return "Factura mixta: se concilian los valores por pagar y por cobrar antes de registrarla.";
    }
}


In [ ]:
%%writefile v2-mixta/src/informes/CompraMixta.java
package informes;

/** Producto concreto: compra MIXTA (recepción y despacho en un mismo documento). */
public class CompraMixta extends InformeBase implements InformeCompra {

    public CompraMixta() {
        super("MIXTA", "COMPRA");
    }

    @Override
    protected String accion() {
        return "Compra mixta: se separan las líneas de recepción y de despacho y se ajusta el inventario.";
    }
}


### 2.2 Clases modificadas (`Categoria`: una línea; `Main`: solo la lista de prueba)

In [ ]:
%%writefile v2-mixta/src/informes/Categoria.java
package informes;

/**
 * Asocia el prefijo de categoría del nombre de archivo (CAT_) con su fábrica
 * concreta. Es el único punto del sistema que conoce las fábricas concretas.
 */
public enum Categoria {

    ENT(new FabricaEntrada()),
    SAL(new FabricaSalida()),
    MIX(new FabricaMixta());

    private final FabricaInformes fabrica;

    Categoria(FabricaInformes fabrica) {
        this.fabrica = fabrica;
    }

    public FabricaInformes getFabrica() {
        return fabrica;
    }

    public static Categoria desdeCodigo(String codigo) {
        for (Categoria categoria : values()) {
            if (categoria.name().equals(codigo)) {
                return categoria;
            }
        }
        throw new IllegalArgumentException("categoría desconocida '" + codigo + "'");
    }
}


In [ ]:
%%writefile v2-mixta/src/informes/Main.java
package informes;

import java.io.File;
import java.util.Arrays;

/**
 * Prueba del sistema. Sin argumentos usa la lista de ejemplo; con una ruta
 * de carpeta procesa los .txt que contiene; con varios argumentos los toma
 * como nombres de informe.
 */
public class Main {

    public static void main(String[] args) {
        String[] reports = {
            "ENT_FAC_001.txt", "ENT_COM_001.txt", "SAL_FAC_001.txt", "SAL_COM_001.txt",
            "MIX_FAC_001.txt", "MIX_COM_001.txt",
            "XYZ_FAC_001.txt", "ENT_FAC.txt"
        };
        if (args.length == 1 && new File(args[0]).isDirectory()) {
            reports = new File(args[0]).list((carpeta, nombre) -> nombre.endsWith(".txt"));
            Arrays.sort(reports);
        } else if (args.length > 0) {
            reports = args;
        }
        new ProcesadorInformes().procesarInformes(reports);
    }
}


### 2.3 Compilación y ejecución

In [ ]:
!cd v2-mixta && rm -rf out && javac -encoding UTF-8 -d out src/informes/*.java && java -cp out informes.Main

Cambios respecto a la versión base:

In [ ]:
!diff -r v1-base/src v2-mixta/src

## 3. Evolución 2: tipo PEDIDO solo en ENTRADA (ejercicio 3b)

PEDIDO es un **producto nuevo** que solo existe en una familia. Se añade a `FabricaInformes` como método `default` que lanza `UnsupportedOperationException`, y solo `FabricaEntrada` lo sobrescribe. `FabricaSalida`, `FabricaMixta` y el cliente **no cambian**; un archivo como `SAL_PED_001.txt` se reporta como omitido.

In [ ]:
!rm -rf v3-pedido && cp -r v2-mixta v3-pedido && rm -rf v3-pedido/out

### 3.1 Clases nuevas

In [ ]:
%%writefile v3-pedido/src/informes/InformePedido.java
package informes;

/** Producto abstracto: informe de tipo PEDIDO (evolución 2; solo existe en ENTRADA). */
public interface InformePedido extends Informe {
}


In [ ]:
%%writefile v3-pedido/src/informes/PedidoEntrada.java
package informes;

/** Producto concreto: pedido de ENTRADA (solicitado por un cliente). */
public class PedidoEntrada extends InformeBase implements InformePedido {

    public PedidoEntrada() {
        super("ENTRADA", "PEDIDO");
    }

    @Override
    protected String accion() {
        return "Pedido recibido de cliente: se verifica la disponibilidad y se reserva el inventario.";
    }
}


### 3.2 Clases modificadas

In [ ]:
%%writefile v3-pedido/src/informes/FabricaInformes.java
package informes;

/**
 * Fábrica abstracta (Abstract Factory). Declara un método de creación por
 * cada tipo de informe; cada fábrica concreta representa una categoría y
 * devuelve la familia completa de informes de esa categoría.
 */
public interface FabricaInformes {

    InformeFactura crearFactura();

    InformeCompra crearCompra();

    /**
     * Evolución 2: el tipo PEDIDO solo existe en la categoría ENTRADA. El
     * método por defecto evita modificar las fábricas que no lo soportan;
     * solo FabricaEntrada lo sobrescribe.
     */
    default InformePedido crearPedido() {
        throw new UnsupportedOperationException("la categoría no admite informes de tipo PEDIDO");
    }
}


In [ ]:
%%writefile v3-pedido/src/informes/FabricaEntrada.java
package informes;

/** Fábrica concreta de la categoría ENTRADA. */
public class FabricaEntrada implements FabricaInformes {

    @Override
    public InformeFactura crearFactura() {
        return new FacturaEntrada();
    }

    @Override
    public InformeCompra crearCompra() {
        return new CompraEntrada();
    }

    @Override
    public InformePedido crearPedido() {
        return new PedidoEntrada();
    }
}


In [ ]:
%%writefile v3-pedido/src/informes/TipoInforme.java
package informes;

import java.util.function.Function;

/**
 * Asocia el código de tipo del nombre de archivo (_TIPO_) con el método de
 * creación correspondiente de la fábrica abstracta.
 */
public enum TipoInforme {

    FAC(FabricaInformes::crearFactura),
    COM(FabricaInformes::crearCompra),
    PED(FabricaInformes::crearPedido);

    private final Function<FabricaInformes, Informe> creador;

    TipoInforme(Function<FabricaInformes, Informe> creador) {
        this.creador = creador;
    }

    public Informe crear(FabricaInformes fabrica) {
        return creador.apply(fabrica);
    }

    public static TipoInforme desdeCodigo(String codigo) {
        for (TipoInforme tipo : values()) {
            if (tipo.name().equals(codigo)) {
                return tipo;
            }
        }
        throw new IllegalArgumentException("tipo de informe desconocido '" + codigo + "'");
    }
}


In [ ]:
%%writefile v3-pedido/src/informes/Main.java
package informes;

import java.io.File;
import java.util.Arrays;

/**
 * Prueba del sistema. Sin argumentos usa la lista de ejemplo; con una ruta
 * de carpeta procesa los .txt que contiene; con varios argumentos los toma
 * como nombres de informe.
 */
public class Main {

    public static void main(String[] args) {
        String[] reports = {
            "ENT_FAC_001.txt", "ENT_COM_001.txt", "SAL_FAC_001.txt", "SAL_COM_001.txt",
            "MIX_FAC_001.txt", "MIX_COM_001.txt", "ENT_PED_001.txt",
            "SAL_PED_001.txt", "XYZ_FAC_001.txt", "ENT_FAC.txt"
        };
        if (args.length == 1 && new File(args[0]).isDirectory()) {
            reports = new File(args[0]).list((carpeta, nombre) -> nombre.endsWith(".txt"));
            Arrays.sort(reports);
        } else if (args.length > 0) {
            reports = args;
        }
        new ProcesadorInformes().procesarInformes(reports);
    }
}


### 3.3 Compilación y ejecución

In [ ]:
!cd v3-pedido && rm -rf out && javac -encoding UTF-8 -d out src/informes/*.java && java -cp out informes.Main

Cambios respecto a la versión con MIXTA:

In [ ]:
!diff -r v2-mixta/src v3-pedido/src